# Sepang counterfactual: what if Hamilton had started on inters?

2026 Bahrain Grand Prix (held at Sepang, Round 16, 4 Oct 2026). Ferrari started Hamilton from P2 on slicks on a wet track. He finished P3.
This notebook runs the whole analysis end to end. Every number comes from `config.yaml` and the code in `src/`.

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # the notebook runs from notebooks/; src/ lives at the repo root
import fastf1; fastf1.set_log_level("ERROR")  # hide download chatter

from src.data_loader import load_config, load_session, fact_check

cfg = load_config()
session = load_session(cfg)

Request for URL https://api.jolpi.ca/ergast/f1/2026/16/results.json failed; using cached response
Traceback (most recent call last):
  File "D:\Git\f1-2026-sepang-counterfactual\.venv\Lib\site-packages\urllib3\connection.py", line 239, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "D:\Git\f1-2026-sepang-counterfactual\.venv\Lib\site-packages\urllib3\util\connection.py", line 85, in create_connection
    raise err
  File "D:\Git\f1-2026-sepang-counterfactual\.venv\Lib\site-packages\urllib3\util\connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
TimeoutError: timed out

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "D:\Git\f1-2026-sepang-counterfactual\.venv\Lib\site-packages\urllib3\connectionpool.py", line 793, in urlopen
    response = self._make_request(
        

Request for URL https://api.jolpi.ca/ergast/f1/2026/16/laps/1.json failed; using cached response
Traceback (most recent call last):
  File "D:\Git\f1-2026-sepang-counterfactual\.venv\Lib\site-packages\urllib3\connection.py", line 239, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "D:\Git\f1-2026-sepang-counterfactual\.venv\Lib\site-packages\urllib3\util\connection.py", line 85, in create_connection
    raise err
  File "D:\Git\f1-2026-sepang-counterfactual\.venv\Lib\site-packages\urllib3\util\connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
TimeoutError: timed out

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "D:\Git\f1-2026-sepang-counterfactual\.venv\Lib\site-packages\urllib3\connectionpool.py", line 793, in urlopen
    response = self._make_request(
        c

## M1: what does the data actually say?

Before modelling anything, check the race story against the timing data. The PRD's rule is that **the data wins**.

In [2]:
fact_check(session, cfg)


=== Event ===
Bahrain Grand Prix | Kuala Lumpur | 2026-10-04 | 55 laps

=== Official classification (top 6 + Ferraris) ===
                     TeamName  GridPosition  Position    Status  gap_s
Abbreviation                                                          
VER           Red Bull Racing           1.0       1.0  Finished  0.000
ANT                  Mercedes           3.0       2.0  Finished  2.307
HAM                   Ferrari           2.0       3.0  Finished  4.919
LEC                   Ferrari           4.0       4.0  Finished  7.258
HAD           Red Bull Racing           8.0       5.0  Finished  8.571
PIA                   McLaren           6.0       6.0  Finished  9.454

=== How the race started (race control) ===
 Lap                                                                                                                         Message
   1                                                                                                                   DELAYED STA

### What the data changed

| PRD / press said | Data says | Why it matters |
|---|---|---|
| Wet race | **No rain fell during the race.** It stopped about 54 min before the start, so the race ran on a drying track | Inter pace improves lap by lap, so a single "inter pace" number is wrong |
| Hamilton P2 → ~P16 on lap 1 | Laps 1–2 were **behind the SC**, then a **standing start**. HAM was P15 after lap 3 (the first racing lap) | The PRD's "lap-1 delta" becomes a lap-3 standing-start delta |
| Long wet phase, then a crossover | Wet phase = **laps 3–8 only**. All the leaders pitted for slicks on **lap 9, under the SC** | The crossover stop was cheap for everyone. The ghost gets the same cheap stop |
| Gamble cost Hamilton the race | He was **83 s** behind after lap 8, but only **6.5 s** after the SC, at P15 | The SC refunded the *time* but not the *track position*. Then he had to pass 10 cars |
| Leclerc P4 | Official P4, but FastF1 timing puts him P17 (+26.8 s) | A per-car timing glitch (+19.5 s from lap 43). We use the official classification for results |

**FastF1 lessons:**
- If the classification API is down, `session.results` quietly falls back to timing-derived positions. `load_session` refuses to continue if that happens.
- Unknown compounds come through as the *string* `'None'`.
- `TrackStatus` doesn't show the SC at the race start.
- Count a lap as SC/VSC only when most cars saw it, because back-markers' laps overlap the leaders' SC.

## M2: race phases and pace models

### Phases
Each lap gets a **track condition** (wet up to the crossover lap, dry after) and a **neutralisation** (green / SC / VSC).
Three independent signals put the crossover on lap 9:
- the leaders' pit stops;
- the slick-minus-inter lap-time gap reaching ~0;
- the SC itself.

In [3]:
from src.segmentation import phase_table
from src.pace_model import add_lap_features, fit_pace, inter_pace, start_gaps
from src.pit_loss import pit_stops, pit_loss_summary

phases = phase_table(session, cfg)
phases.loc[1:20]

,condition,neutral,crossover,slick_minus_inter_s,field_median_s
LapNumber,,,,,
1,wet,sc,False,NaN,NaN
2,wet,sc,False,NaN,NaN
3,wet,green,False,26.57,116.35
4,wet,green,False,21.02,113.00
5,wet,green,False,14.25,112.85
6,wet,green,False,10.33,113.32
7,wet,green,False,4.71,114.21
8,wet,green,False,0.48,113.93
9,wet,sc,True,NaN,NaN


`field_median_s` keeps falling until lap 16, so **the track was still drying after the crossover**.
That's why the dry fit starts at lap 17 (`pace.dry_fit_start_lap`). Before that, track drying would be mistaken for tyre wear.

### Clean laps
A lap is usable for pace only if it is:
- not an in-lap or out-lap;
- not behind the SC/VSC, and not lap 1–3;
- at least `pace.traffic_gap_s` behind the car ahead at the line (lapped or not);
- not flagged inaccurate by FastF1, and not deleted for track limits.

**Effective tyre age** counts only green dry laps as wear (`pace.wet_sc_wear` = 0). So HAM's softs, used in the wet from lap 1, only start ageing at lap 13.

In [4]:
df = add_lap_features(session.laps, phases, cfg)
print(f"clean laps: {int(df.clean.sum())} of {len(df)}")
df[df.Driver == "HAM"].set_index("LapNumber")[["lap_s", "Compound", "TyreLife", "eff_age", "gap_ahead", "clean"]].loc[[8, 13, 17, 25, 30, 33, 52, 55]].round(2)

clean laps: 562 of 1145


,lap_s,Compound,TyreLife,eff_age,gap_ahead,clean
LapNumber,,,,,,
8,112.88,SOFT,8.0,0.0,4.39,True
13,110.98,SOFT,13.0,1.0,0.49,False
17,102.78,SOFT,17.0,5.0,0.46,False
25,101.73,SOFT,25.0,13.0,10.14,True
30,102.50,SOFT,30.0,18.0,5.09,True
33,100.18,MEDIUM,5.0,5.0,0.83,False
52,99.62,SOFT,12.0,4.0,0.29,False
55,99.44,SOFT,15.0,7.0,2.60,True


### Dry pace: one fit over the whole field

`lap_s = <compound> + deg_<compound> * eff_age + remaining * laps_remaining + d_<driver>`, with HAM as the reference (d = 0).

**Why the whole field?** A HAM + LEC fit gave deg 0.19 s/lap and LEC 1.1 s slower. Comparing HAM's laps 17–30 with 52–55 mixes three things: tyre wear, fuel burn and the track rubbering in.
On any single lap, 20 cars run very different tyre ages. That's what separates deg from `remaining` (fuel + track, *measured* at ~0.044 s/lap).
`d_<drv>` is each driver's offset to HAM (positive = slower).

In [5]:
fit = fit_pace(df, cfg)
ip = inter_pace(df, cfg)
table = pd.DataFrame({"coef": fit["coef"], "se": np.sqrt(np.diag(fit["cov"]))}).round(3)
table[~table.index.str.startswith("d_") | table.index.isin([f"d_{d}" for d in ["LEC"] + ip["leaders"]])]

,coef,se
HARD,98.918,0.297
MEDIUM,99.494,0.336
SOFT,98.791,0.233
UNKNOWN,98.476,1.010
deg_HARD,0.184,0.013
deg_MEDIUM,0.111,0.021
deg_SOFT,0.148,0.015
deg_UNKNOWN,0.106,0.074
remaining,0.044,0.004
d_ANT,-0.256,0.237


### Inter pace for HAM, who never ran inters

HAM's inter pace is modelled as an offset (s/lap) to `ref`, the leaders' median clean inter lap on each wet lap. `ref` follows the drying track.
- **Proxy**: LEC's inter laps minus `ref`, then adjusted by the HAM−LEC dry gap.
- **Ratio**: HAM's dry gap to the leaders, scaled by the wet/dry lap-length ratio.

The two disagree by about 1.3 s/lap. The proxy is pulled faster for two reasons:
- LEC's inters were **3 laps fresher** than the leaders' on a drying track;
- LEC was **~1 s/lap slower in the dry** (robust to the traffic filter; possibly damage from his lap-3 collision), and that gap gets added back.

So the Monte Carlo samples Normal(mu, sigma), with sigma wide enough to cover both. `pace.proxy_weight` sets mu. The M5 tornado tests it.

In [6]:
print("reference:", ip["ref"].round(2).to_dict())
print(f"proxy {ip['proxy']:+.2f} +/- {ip['proxy_se']:.2f} | ratio {ip['ratio']:+.2f} +/- {ip['ratio_se']:.2f} "
      f"| Normal(mu={ip['mu']:+.2f}, sigma={ip['sigma']:.2f}) s/lap")
start_gaps(df, session.results, cfg).round(2)

reference: {4: 109.08, 5: 110.46, 6: 111.37, 7: 112.65, 8: 112.62}
proxy -1.55 +/- 0.41 | ratio -0.23 +/- 0.20 | Normal(mu=-0.89, sigma=0.74) s/lap


,grid,pos,gap_s
Driver,,,
VER,1.0,3.0,2.26
ANT,3.0,1.0,0.00
RUS,7.0,2.0,0.87
HAD,8.0,5.0,5.14


### Pit loss

Loss = (in-lap − ref) + (out-lap − ref).
- **Green stops:** the reference is the driver's own clean pace nearby. The median comes out at about 26.5 s.
- **Neutralised stops:** the reference is the median lap of the cars that didn't pit.

That reference **breaks when the SC comes out mid-lap** (laps 9 and 43). Cars that had already done most of the lap at racing speed look cheap (VER ~10 s); cars further back look expensive (~24 s). The loss depends on *where you were* when the SC came out.
So M3 won't use a single SC number. For a stop on a neutralised lap, the ghost copies the in-lap and out-lap of cars that stopped on that same lap from a similar position.

In [7]:
stops = pit_stops(df, cfg)
pit_loss_summary(stops)

,n,median_s,min_s,max_s
neutral,,,,
green,28,26.54,23.38,52.05
sc,31,18.29,4.95,24.68
vsc,7,21.28,1.11,46.20
